In [ ]:
from nltk.corpus import propbank
import json
from urllib.request import urlopen
import numpy as np
import urllib
import os
from os import chdir
from nltk.corpus import treebank
from nltk.tree import Tree
import itertools
import re
EXPERIMENT_NAME = 'SRL'

**Cloning to PropBank Release resource**

In [ ]:
!git clone https://github.com/propbank/propbank-release.git
!ls
%cd propbank-release/data/ontonotes/nw/wsj

In [ ]:
os.getcwd()

**Access to propositions through .prop file extension.....**

In [ ]:
prefix = "wsj"
folder_num = ['00','01','02','03','04','05','06','07','08','09','10','11','12','13','14','15','16','17','18','19','20','21','22','23','24']
file_num = []
for sis in range(1,2455):
  file_num.append(str(f"{sis:04}"))

sample = []
existing_files = []
for folder in folder_num:
  %cd '{folder}'
  path = os.getcwd()
  for sub_file in file_num:
    lines = []
    if(os.path.exists(f'{path}/{prefix}_{sub_file}.prop')):
      with open(f'{prefix}_{sub_file}.prop', 'r') as file:
           existing_files.append(f'{prefix}_{sub_file}.onf')
           data = file.read()
           sample.append(data)
  %cd ..

In [ ]:
sample[0]

In [ ]:
new_list = []
for i in range(len(sample)):
  x = ''
  x = sample[i].split("\n")
  x.append('')
  new_list.extend(x)

In [ ]:
new_list

**Sorting all existing proposition strings within each .prop file based on their prioritization.**

In [ ]:
import itertools

w = '' #within sample at the end of each wsj file there is a empety line ('')

split_annotations = [list(y) for x, y in itertools.groupby(new_list, lambda z: z == w) if not x]

In [ ]:
split_annotations

In [ ]:
# helper function to perform sort

def num_sort(test_string):
    return list(map(int, re.findall(r'\d+', test_string)))[2:4]

for i in range(len(split_annotations)):
  split_annotations[i].sort(key=num_sort)

In [ ]:
split_annotations

***Note: There is a discrepancy between the existing files in the wsj files and the split_annotations list. To establish a correct approach, we removed four indices (or files) from both the existing files and the split_annotations list.***

In [ ]:
del split_annotations[112]
del split_annotations[612]
del split_annotations[617]
del split_annotations[621]

del existing_files[112]
del existing_files[612]
del existing_files[617]
del existing_files[621]

In [ ]:
len(existing_files)

***We have defined an index list that helps us connect each raw annotation string to its corresponding parse tree, enabling conversion into its textual spans.***

In [ ]:
index = []
for i in range(len(split_annotations)):
  lst = []
  for n in range(len(split_annotations[i])):
    lst.append(int(split_annotations[i][n].split()[1]))
  index.append(list(lst))

In [ ]:
index

In [ ]:
# For example, the file wsj_0001.prop contains two sentences. The first sentence has only one predicate,
#while the second sentence has two predicates. The index list clearly maps the parse tree indices, which
#are represented as 0 and 1.

print(split_annotations[0])
print("/**************************************/")
print(index[0])

**Change directory to the OntoNotes 5.0 Corpus**

In [ ]:
%cd ..
%cd ..
%cd ..
%cd ..
%cd ..

In [ ]:
# Dataset placeholders
data_name_to_google_drive_url = {
    '1': '{{PLACEHOLDER_ONTONOTES_5_0_URL}}',
    '2': '{{PLACEHOLDER_PROPBANK_RELEASE_URL}}'
}

# Get direct download link
def get_download_url_from_google_drive_url(google_drive_url):
    if 'drive.google.com' in google_drive_url:
        return f'https://drive.google.com/uc?id={google_drive_url.split("/")[5]}&export=download&confirm=t'
    return google_drive_url

In [ ]:
# Temporary direct download URL placeholders
data_temp_url = {
    '1': '{{PLACEHOLDER_ONTONOTES_DOWNLOAD_LINK}}',
    '2': '{{PLACEHOLDER_PROPBANK_DOWNLOAD_LINK}}'
}

In [ ]:
for data_name in ['1', '2']:
  google_drive_url = data_name_to_google_drive_url[data_name]
  data_url = get_download_url_from_google_drive_url(google_drive_url)
  print(data_url)
  data_url = data_temp_url[data_name]
  urllib.request.urlretrieve(data_url, "response.zip")
  os.system('unzip response.zip')

In [ ]:
%cd LDC2013T19-ontonotes-release-5.0/data/files/data/english/annotations/nw/wsj

In [ ]:
os.getcwd()

**Extraction of Plain (Standard) Sentences from the files with .onf extension.**

In [ ]:
prefix = "wsj"
folder_num = ['00','01','02','03','04','05','06','07','08','09','10','11','12','13','14','15','16','17','18','19','20','21','22','23','24']

pmp = []
sent = []
for folder in folder_num:
  %cd '{folder}'
  path = os.getcwd()
  for file_num in existing_files:
    lines = []
    if(os.path.exists(f'{path}/{file_num}')):
      with open(f'{file_num}', 'r') as file:
           data = file.read()
           lines = data.split('\n\n')
           for i in range(0,len(lines)):
             if (lines[i] == '\n------------------------------------------------------------------------------------------------------------------------' or lines[i] == '------------------------------------------------------------------------------------------------------------------------'):
               sent.append( (lines[i+1].split("\n---------------\n    ")[1]).replace("\n   ", "") )
           sent.append("")
  %cd ..

In [ ]:
sent

In [ ]:
w = '' #within sample at the end of each wsj file there is a empety line ('')

split_standard_sentences = [list(y) for x, y in itertools.groupby(sent, lambda z: z == w) if not x]

In [ ]:
split_standard_sentences

In [ ]:
len(split_standard_sentences)

**Extraction of Treenanked Sentences (with Trace Anchors) from the files with .onf extension.**

In [ ]:
prefix = "wsj"
folder_num = ['00','01','02','03','04','05','06','07','08','09','10','11','12','13','14','15','16','17','18','19','20','21','22','23','24']

pmp = []
sent_anchor = []
for folder in folder_num:
  %cd '{folder}'
  path = os.getcwd()
  for file_num in existing_files:
    lines = []
    if(os.path.exists(f'{path}/{file_num}')):
      with open(f'{file_num}', 'r') as file:
           data = file.read()
           lines = data.split('\n\n')
           for i in range(0,len(lines)):
             if (lines[i] == '\n------------------------------------------------------------------------------------------------------------------------' or lines[i] == '------------------------------------------------------------------------------------------------------------------------'):
               sent_anchor.append( (lines[i+2].split("\n--------------------\n    ")[1]).replace("\n   ", "") )
           sent_anchor.append("")
  %cd ..

In [ ]:
sent_anchor

In [ ]:
w = '' #within sample at the end of each wsj file there is a empety line ('')

split_sentences_with_anchor = [list(y) for x, y in itertools.groupby(sent_anchor, lambda z: z == w) if not x]

In [ ]:
split_sentences_with_anchor

In [ ]:
len(split_sentences_with_anchor)

**Parse Tree Extraction using .parse files.**

In [ ]:
prefix = "wsj"
folder_num = ['00','01','02','03','04','05','06','07','08','09','10','11','12','13','14','15','16','17','18','19','20','21','22','23','24']

pmp = []
tree = []
for folder in folder_num:
  %cd '{folder}'
  path = os.getcwd()
  for file_num in existing_files:
    lines = []
    if(os.path.exists(f'{path}/{file_num}')):
      with open(f'{file_num}', 'r') as file:
           data = file.read()
           lines = data.split('\n\n')
           for i in range(0,len(lines)):
             if (lines[i] == '\n------------------------------------------------------------------------------------------------------------------------' or lines[i] == '------------------------------------------------------------------------------------------------------------------------'):
               tree.append( (lines[i+3].split("Tree:\n-----\n    (TOP ")[1])[:-1] )
           tree.append("")
  %cd ..

In [ ]:
tree

In [ ]:
w = '' #within sample at the end of each wsj file there is a empety line ('')

split_trees = [list(y) for x, y in itertools.groupby(tree, lambda z: z == w) if not x]

In [ ]:
split_trees

In [ ]:
len(split_trees)

**Index list as a connector between split_trees & split_annotations**

In [ ]:
_annotations_ = []
_tree_ = []
_plain_sentence_ = []
_treebanked_sentence_ = []
for i in range(len(split_annotations)):
  for n in range(len(split_annotations[i])):
    _annotations_.append(split_annotations[i][n])
    _tree_.append(split_trees[i][index[i][n]])
    _plain_sentence_.append(split_standard_sentences[i][index[i][n]])
    _treebanked_sentence_.append(split_sentences_with_anchor[i][index[i][n]])

In [ ]:
_annotations_

In [ ]:
_tree_

In [ ]:
_plain_sentence_

In [ ]:
_treebanked_sentence_

**Extraction of ARG0 & ARG1 Annotations**

In [ ]:
arg_zero = []
for i in range (0,131287):
  arg_zero.append("")

for i in range(len(_annotations_)):
  list0 = []
  list0.extend(_annotations_[i].split())
  for s in range(len(list0)):
    if(len(list0[s]) > 4 and list0[s][-1] == "0" and list0[s][-2] == "G" and list0[s][-3] == "R" and list0[s][-4] == "A"):
      arg_zero[i] = list0[s].split("-")[0]

In [ ]:
arg_one = []
for i in range (0,131287):
  arg_one.append("")

for i in range(len(_annotations_)):
  list1 = []
  list1.extend(_annotations_[i].split())
  for s in range(len(list1)):
    if(len(list1[s]) > 4 and list1[s][-1] == "1" and list1[s][-2] == "G" and list1[s][-3] == "R" and list1[s][-4] == "A"):
      arg_one[i] = list1[s].split("-")[0]

**Extraction of predicate Annotations**

In [ ]:
rel = []
for i in range (0,131287):
  rel.append("")

for i in range(len(_annotations_)):
  list2 = []
  list2.extend(_annotations_[i].split())
  for s in range(len(list2)):
    if(len(list2[s]) > 4 and list2[s][-1] == "l" and list2[s][-2] == "e" and list2[s][-3] == "r"):
      rel[i] = list2[s].split("-")[0]

In [ ]:
arg_zero

In [ ]:
arg_one

In [ ]:
rel

**Conversion of Annotations into Text Spans**

In [ ]:
from nltk.corpus.reader.propbank import PropbankTreePointer
from nltk.tree import Tree

In [ ]:
# Create a copy of each one of annotations

ARG0_real = arg_zero.copy()
ARG1_real = arg_one.copy()
REL_real = rel.copy()

**Important Function**

In [ ]:
def convertor(string):
  delimiters = ["*", ",", ";"]
  for delimiter in delimiters:
    string = " ".join(string.split(delimiter))
  result = string.split()
  return result

In [ ]:
def parse_tree_to_text(parse_tree_str):
    # Convert string representation of parse tree to NLTK Tree object
    parse_tree = Tree.fromstring(parse_tree_str)

    # Extract words from the parse tree
    words = parse_tree.leaves()

    # Join words to form a sentence
    text = ' '.join(words)

    return text

In [ ]:
def TreePointer(x):
  lst_2 = []
  lst_2.extend(re.split(';|:', x))
  pointer = PropbankTreePointer(int(lst_2[0]),int(lst_2[1]))
  pointer
  return pointer

**Conversion of Raw Annotations into Split Annotations.....**

In [ ]:
for i in range(len(arg_zero)):
  if (arg_zero[i] == ''):
    ARG0_real[i] = arg_zero[i]
  else:
    ARG0_real[i] = convertor(arg_zero[i])

In [ ]:
for i in range(len(arg_one)):
  if (arg_one[i] == ''):
    ARG1_real[i] = arg_one[i]
  else:
    ARG1_real[i] = convertor(arg_one[i])

In [ ]:
for i in range(len(rel)):
  if (rel[i] == ''):
    REL_real[i] = rel[i]
  else:
    REL_real[i] = convertor(rel[i])

In [ ]:
ARG0_real

In [ ]:
ARG0 = ARG0_real.copy()
ARG1 = ARG1_real.copy()
REL = REL_real.copy()

**Solving a problem in some parse trees for the lack of finishing ")"**

In [ ]:
z = []
for i in range(len(_tree_)):
  if (_tree_[i] == '(S (ADVP (RB Still))\n            (, ,)\n            (NP-SBJ (NNP Westinghouse))\n            (VP (VBZ acknowledges)\n                (SBAR (IN that)\n                      (S (NP-SBJ (NP (NN demand))\n                                 (PP (IN from)\n                                     (NP (JJ independent)\n                                         (NNS producers))))\n                         (VP (MD could)\n                             (VP (VB evaporate)\n                                 (SBAR-ADV (SBAR (IN if)\n                                                 (S (NP-SBJ (NP (NNS prices))\n                                                            (PP (IN for)\n                                                                (NP (NP (NN fuel))\n                                                                    (PP (JJ such)\n                                                                        (IN as)\n                                                                        (NP (NP (JJ natural)\n                                                                                (NN gas))\n                                                                            (CC or)\n                                                                            (NP (NN oil)))))))\n                                                    (VP (VBP rise)\n                                                        (ADVP-MNR (RB sharply)))))\n                                           (CC or)\n                                           (SBAR (IN if)\n                                                 (S (NP-SBJ-3 (NP (NNS utilities))\n                                                              (, ,)\n                                                              (SBAR (WHNP-2 (WDT which))\n                                                                    (S (NP-SBJ-1 (-NONE- *T*-2))\n                                                                       (VP (VBP have)\n                                                                           (VP (VBN been)\n                                                                               (VP (VBN pressured)\n                                                                                   (PP (IN by)\n                                                                                       (NP-LGS (NNS regulators)))\n                                                                                   (NP-4 (-NONE- *-1))\n                                                                                   (S (NP-SBJ (-NONE- *PRO*-4))\n                                                                                      (VP (TO to)\n                                                                                          (VP (VB keep)\n                                                                                              (ADVP-CLR (RB down)))\n                                                                                          (NP (NNS rates))))))))))\n                                                    (, ,))\n                                                 (VP (VBP are)\n                                                     (ADVP-TMP (RB suddenly))\n                                                     (VP (VBN freed)\n                                                         (S (NP-SBJ (-NONE- *PRO*-3))\n                                                            (VP (TO to)\n                                                                (VP (VB add)\n                                                                    (NP (JJ significant)\n                                                                        (VBG generating)\n                                                                        (NN capacity)))))))))))))))\n         (. .)'):
    z.append(i)

In [ ]:
z         #indices of parse trees

In [ ]:
s = list(_tree_[29809] + ")")
s[3771] = ''
_tree_[29809] = "".join(s)
_tree_[29810] = "".join(s)
_tree_[29811] = "".join(s)
_tree_[29812] = "".join(s)
_tree_[29813] = "".join(s)
_tree_[29814] = "".join(s)
_tree_[29815] = "".join(s)
_tree_[29816] = "".join(s)
_tree_[29817] = "".join(s)
_tree_[29818] = "".join(s)
_tree_[29819] = "".join(s)

**Conversion of Split Annotations into Textual Spans**

In [ ]:
for i in range(len(ARG0_real)):
  for m in range(len(ARG0_real[i])):
    if (ARG0_real[i][m] == ''):
      ARG0[i][m] = ARG0_real[i][m]
    else:
      ARG0[i][m] = parse_tree_to_text(str(TreePointer(ARG0_real[i][m]).select(Tree.fromstring(_tree_[i]))))

In [ ]:
for i in range(len(ARG1_real)):
  for m in range(len(ARG1_real[i])):
    if (ARG1_real[i][m] == ''):
      ARG1[i][m] = ARG1_real[i][m]
    else:
      ARG1[i][m] = parse_tree_to_text(str(TreePointer(ARG1_real[i][m]).select(Tree.fromstring(_tree_[i]))))

In [ ]:
for i in range(len(REL_real)):
  for m in range(len(REL_real[i])):
    if (REL_real[i][m] == ''):
      REL[i][m] = REL_real[i][m]
    else:
      REL[i][m] = parse_tree_to_text(str(TreePointer(REL_real[i][m]).select(Tree.fromstring(_tree_[i]))))

In [ ]:
ARG0

In [ ]:
ARG1

In [ ]:
REL

**Creation of Final lists through Joining Textual Spans .......................**

In [ ]:
list_sentences = []
list_sentences_with_anchor = []
list_predicates = []
list_ARG0s = []
list_ARG1s = []

for i in range(len(_plain_sentence_)):
  list_sentences.append(_plain_sentence_[i])
  list_sentences_with_anchor.append(_treebanked_sentence_[i])
  list_predicates.append(' '.join(REL[i]))
  list_ARG0s.append(' '.join(ARG0[i]))
  list_ARG1s.append(' '.join(ARG1[i]))

In [ ]:
list_sentences

In [ ]:
list_sentences_with_anchor

In [ ]:
list_predicates

In [ ]:
list_ARG0s

In [ ]:
list_ARG1s

In [ ]:
print(len(list_sentences))
print(len(list_sentences_with_anchor))
print(len(list_predicates))
print(len(list_ARG0s))
print(len(list_ARG1s))

**Initial DataFrame**

In [ ]:
# import pandas as pd
import pandas as pd

# Calling DataFrame constructor after zipping
# both lists, with columns specified
df = pd.DataFrame(list(zip(list_sentences, list_sentences_with_anchor, list_predicates, list_ARG0s, list_ARG1s)),
               columns =['SENTENCE' ,	'SENTENCE_ANCHOR' ,	'PREDICATES' ,	'ARG0' , 'ARG1'])

In [ ]:
df

**Regular Expresions for removing trace anchors from ARG0s, ARG1s, and Treebanked sentence.**

In [ ]:
import re

def standardize_text(sentence):
  pattern = r"0|[*][A-Z]+[*]-\d+|[*][A-Z]+[*]|[*][?][*]|[*]-\d+|[*]"
  cleaned_sentence = re.sub(pattern, '', sentence).strip()
  cleaned_sentence = re.sub(r"\s+", ' ',cleaned_sentence)

  return cleaned_sentence

In [ ]:
x = "The *PRO*-2 Sacramento - based S&L , which *T*-110 has 44 branch offices *?* in north * *PRO* central California , *-1 had assets of $ 2.4 billion *U* at the end of September *NOT* ."

In [ ]:
standardize_text(x)

In [ ]:
sent = []
pred = []
arg0 = []
arg1 = []
for i in range(0,131287):
  sent.append(standardize_text(df["SENTENCE_ANCHOR"][i]))
  pred.append(standardize_text(df["PREDICATES"][i]))
  arg0.append(standardize_text(df["ARG0"][i]))
  arg1.append(standardize_text(df["ARG1"][i]))

In [ ]:
arg_merg = []
for m in range(0,131287):
  arg_merg.append(arg0[m] + "|" + arg1[m])

In [ ]:
arg_merg

In [ ]:
df_2 = pd.DataFrame(list(zip(sent , pred , arg0, arg1 , arg_merg)),
               columns =['SENTENCE' ,	'PREDICATES' ,	'ARG0' , 'ARG1' , 'Merged_Arguments'])

**Removing observations with both null ARG0 and ARG1**

In [ ]:
# Finding the index of sample which does not have any arguments
num = []
for i in range(0, 131287):
  if (arg_merg[i] == "|"):
    num.append(i)

# Finding the relative predicate & sentence with indices
predicates_without_arguments = []
sentences_ll = []
for i in num:
  predicates_without_arguments.append(df_2["PREDICATES"][i])
  sentences_ll.append(df_2["SENTENCE"][i])

In [ ]:
final_SRL_DATASET = df_2.drop(num).reset_index(drop=True)

In [ ]:
final_SRL_DATASET

In [ ]:
from google.colab import files

final_SRL_DATASET.to_csv('df.csv')
files.download('df.csv')

In [ ]:
# Here our operations is finished, but for extracting the existing observation of OntoNotes 5.0 Corpus
# We follow repeated steps for the ongoing objective in the second notebook of data extraction folder
############################################################################################################
    #################################################################################################
        ########################################################################################